# Binary ARCiS Retrieval

This notebook shows the minimal setup for a binary retrieval with `ARCiS_binary`. FlopPITy samples one parameter vector containing both components. `ARCiS_binary` splits that vector into two equal blocks, runs ARCiS once per component, and sums the returned spectra.

The example assumes both components use the same ARCiS input file and therefore the same fitted-parameter list. The parameter names in FlopPITy are suffixed with `_1` and `_2` only so the posterior plots are readable; ARCiS receives the correct component-sized parameter grid by column order.

In [ ]:
from copy import deepcopy

from floppity import Retrieval
from floppity.simulators import ARCiS_binary, read_ARCiS_input

## Paths

Edit these paths for your local ARCiS setup.

In [ ]:
arcis_input = "path/to/arcis_input.in"
arcis_executable = "/usr/local/bin/ARCiS"

arcis_output_dir = "output_ARCiS_binary"
retrieval_output_dir = "output_FlopPITy_binary"

## Read The Single-Component ARCiS Setup

`read_ARCiS_input` returns the fitted parameters and observation files from one ARCiS input file. For a binary retrieval, we duplicate the parameter metadata once per component.

In [ ]:
single_component_pars, obs_dict = read_ARCiS_input(arcis_input)

def make_component_parameters(base_parameters, n_components=2):
    component_parameters = {}
    for component in range(1, n_components + 1):
        for name, metadata in base_parameters.items():
            component_parameters[f"{name}_{component}"] = deepcopy(metadata)
    return component_parameters

binary_parameters = make_component_parameters(single_component_pars, n_components=2)

print(f"Single-component parameters: {len(single_component_pars)}")
print(f"Binary retrieval parameters: {len(binary_parameters)}")
print(list(binary_parameters)[:5])

## Build The Retrieval

The only simulator-specific change is using `ARCiS_binary` instead of `ARCiS`.

In [ ]:
R = Retrieval(ARCiS_binary, obs_type="emis")
R.parameters = binary_parameters
R.get_obs(obs_dict)

# Optional preprocessing. Keep this disabled until you are sure your spectra
# are strictly positive after ARCiS and FlopPITy's emission clipping.
# R.preprocessing = ["log"]

## Runtime Options

Start small while testing that ARCiS runs and the parameter ordering is correct. Increase `n_samples`, `n_samples_init`, and `n_rounds` for a real retrieval.

In [ ]:
ARCiS_kwargs = {
    "input_file": arcis_input,
    "output_dir": arcis_output_dir,
    "ARCiS_dir": arcis_executable,
    "num_threads": "4",
    "save_atmosphere": True,
}

flow_kwargs = {
    "flow": "nsf",
    "bins": 4,
    "transforms": 10,
    "blocks": 3,
    "hidden": 64,
    "dropout": 0.1,
}

training_kwargs = {
    "stop_after_epochs": 10,
    "num_atoms": 20,
    "learning_rate": 1e-3,
}

## Run

`ARCiS_binary` directly sums the two component spectra. If your binary model needs radius weighting, flux ratios, dilution, or another combination rule, add those terms to the simulator wrapper before using this for production science.

In [ ]:
R.run(
    flow_kwargs=flow_kwargs,
    training_kwargs=training_kwargs,
    simulator_kwargs=ARCiS_kwargs,
    output_dir=retrieval_output_dir,
    n_threads=2,
    n_rounds=2,
    n_samples_init=16,
    n_samples=16,
    save_data=True,
    sample_prior_method="sobol",
)

## Inspect The Result

After the run, the posterior contains the full binary parameter vector. The first component's parameters have `_1` suffixes and the second component's parameters have `_2` suffixes.

In [ ]:
fig = R.plot_corner(n_samples=2000)